# 🎯 Multi-Stage Audience Subscription Relevance Filtering & Niche Channel Identification
### *Iterative LLM Batch Reduction, Context-Aware Relevance Ranking, and Channel Selection via Gemini 3.1 Flash-Lite*

---

## 📌 Executive Summary & Analytical Methodology

### Abstract
This Jupyter Notebook implements a multi-stage qualitative filtering and ranking pipeline using **Gemini 3.1 Flash-Lite** to isolate the **10 most relevant YouTube channels** from the cross-channel audience subscription pool exported in previous analytical modules (`4.Youtube_channel_subscription_clusters.ipynb`). Utilizing target channel thematic guidelines (`context/channel_description.txt`), the pipeline executes iterative batch filtering ($N_0 	o N_1 \dots 	o 25 	o 10$) by examining channel video titles across top performing/recent long-form uploads.

### Analytical Pipeline & Mathematical Formulation
Let $C = \{c_1, c_2, \dots, c_M\}$ be the initial pool of audience-subscribed channels imported from exported dataset payloads. For each channel $c_i$, we extract its top 10 best performing videos (ranked by view count/engagement rate) or recent uploads $V(c_i) = \{v_{i,1}, v_{i,2}, \dots, v_{i,10}\}$.

1. **Top Video Selection**:
   $$V^*(c_i) = 	ext{ArgTop}_{k=10} \{ 	ext{Views}(v) \mid v \in V(c_i) \}$$

2. **Iterative Batch Filtering ($10 	o 5$)**:
   The current channel pool $C^{(t)}$ is partitioned into disjoint batches of size $B=10$:
   $$C^{(t)} = B_1^{(t)} \cup B_2^{(t)} \cup \dots \cup B_k^{(t)}, \quad |B_j^{(t)}| \le 10$$
   For each batch $B_j^{(t)}$, Gemini 3.1 Flash-Lite evaluates video titles against the reference channel description $D_{	ext{target}}$ and selects the 5 most relevant channels:
   $$\mathcal{F}_{	ext{LLM}}(B_j^{(t)}, D_{	ext{target}}) 	o S_j^{(t)}, \quad |S_j^{(t)}| = \min(5, |B_j^{(t)}|)$$
   The surviving pool $C^{(t+1)} = igcup_j S_j^{(t)}$ is then randomly shuffled ($\sigma(C^{(t+1)})$). This process repeats iteratively until $|C^{(t+1)}| \le 25$.

3. **Final Top-10 Selection ($25 	o 10$)**:
   When the channel pool size reaches $M_{	ext{final}} \le 25$, a final holistic LLM synthesis query ranks all candidate channels and extracts the **Top 10 channels** with detailed relevance justifications:
   $$\mathcal{F}_{	ext{LLM}}^{	ext{Final}}(C^{	ext{final}}, D_{	ext{target}}) 	o \{c_{(1)}^*, c_{(2)}^*, \dots, c_{(10)}^*\}$$

### Exported Dataset Payloads
- **CSV Payload**: `{Channel_Name}_top_10_relevant_channels.csv`
- **JSON Payload**: `{Channel_Name}_top_10_relevant_channels.json`
- **Markdown Report**: `{Channel_Name}_top_10_relevant_channels.md`


## ⚙️ 1. Configuration & Setup

Initializes global configuration parameters, target model specification (`gemini-3.1-flash-lite`), batch size ($B=10$), retention count per batch ($K_{	ext{batch}}=5$), final stage threshold ($M_{	ext{final}}=25$), and top exported channels ($K_{	ext{top}}=10$). Loads API keys from Google Colab Secrets (`YOUTUBE_API_KEY`, `GOOGLE_API_KEY`) or environment variables.


In [ ]:
# Install required dependencies if missing
!pip install -q google-genai pandas matplotlib seaborn numpy

import os
import re
import time
import json
import random
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Global Configuration Dictionary
CONFIG = {
    "CHANNEL_IDENTIFIER": "@josemillanastrologohumanista",
    "GEMINI_MODEL": "gemini-3.1-flash-lite",
    "BATCH_SIZE": 10,
    "RETENTION_PER_BATCH": 5,
    "FINAL_STAGE_THRESHOLD": 25,
    "FINAL_TOP_K": 10,
    "TOP_VIDEOS_PER_CHANNEL": 10,
    "MAX_RETRIES": 3,
    "RETRY_BACKOFF_FACTOR": 2,
    "DRIVE_OUTPUT_DIR": "/content/drive/MyDrive/YouTube_Analytics",
    "MOUNT_DRIVE": True,
    "USE_CACHE": True,
    "RANDOM_SEED": 42
}

# Set random seed for reproducible shuffling across runs
random.seed(CONFIG["RANDOM_SEED"])

# Obtain Google API Credentials
GOOGLE_API_KEY = None
try:
    from google.colab import userdata
    GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
    if not GOOGLE_API_KEY:
        GOOGLE_API_KEY = userdata.get("YOUTUBE_API_KEY")
except ImportError:
    pass

if not GOOGLE_API_KEY:
    GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY") or os.environ.get("YOUTUBE_API_KEY")

print(f"⚙️ Configuration initialized.")
print(f"  - Model: {CONFIG['GEMINI_MODEL']}")
print(f"  - Batch Size: {CONFIG['BATCH_SIZE']} -> Retain: {CONFIG['RETENTION_PER_BATCH']}")
print(f"  - Final Stage Pool: <= {CONFIG['FINAL_STAGE_THRESHOLD']} -> Final Top K: {CONFIG['FINAL_TOP_K']}")
print(f"  - API Key Present: {bool(GOOGLE_API_KEY)}")


## 💾 2. Mount Google Drive & Storage Hierarchy

Mounts Google Drive at `/content/drive/MyDrive/YouTube_Analytics/` with fallback to local paths (`/content/drive/MyDrive/YouTube_Analytics`, `data`, `analysis`, `.`) to ensure seamless execution across Google Colab and local environments.


In [ ]:
ACTIVE_OUTPUT_DIR = "."

if CONFIG["MOUNT_DRIVE"]:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        os.makedirs(CONFIG["DRIVE_OUTPUT_DIR"], exist_ok=True)
        ACTIVE_OUTPUT_DIR = CONFIG["DRIVE_OUTPUT_DIR"]
        print(f"✅ Google Drive mounted successfully at: {ACTIVE_OUTPUT_DIR}")
    except Exception as e:
        print(f"⚠️ Drive mount unavailable ({e}). Fallback to local working directory: {ACTIVE_OUTPUT_DIR}")
else:
    print(f"ℹ️ Storage initialized locally at: {ACTIVE_OUTPUT_DIR}")


## 🔍 3. LLM Client Initialization, Retry Mechanics & Persistent Disk Caching

Initializes the `google-genai` SDK for `gemini-3.1-flash-lite`. Implements persistent disk caching (`llm_generation_cache.json`) synchronized across Google Drive and local directories, alongside a 3-attempt exponential backoff retry loop (`time.sleep(2 * attempt)`). If requests fail after all retries, explicit exceptions are raised (no synthetic or placeholder data).


In [ ]:
def sanitize_filename(name: str) -> str:
    return re.sub(r'[^a-zA-Z0-9_\-]', '_', name).strip('_')

CACHE_FILE_NAME = "llm_generation_cache.json"

def get_cache_filepath():
    for search_dir in [ACTIVE_OUTPUT_DIR, "/content/drive/MyDrive/YouTube_Analytics", "data", "analysis", "."]:
        if os.path.exists(search_dir):
            candidate = os.path.join(search_dir, CACHE_FILE_NAME)
            if os.path.exists(candidate):
                return candidate
    return os.path.join(ACTIVE_OUTPUT_DIR, CACHE_FILE_NAME)

def load_llm_cache() -> dict:
    cache_path = get_cache_filepath()
    if os.path.exists(cache_path):
        try:
            with open(cache_path, 'r', encoding='utf-8') as f:
                return json.load(f)
        except Exception as e:
            print(f"⚠️ Warning loading cache from {cache_path}: {e}")
    return {}

def save_llm_cache(cache_data: dict):
    target_dirs = [ACTIVE_OUTPUT_DIR]
    if os.path.exists("/content/drive/MyDrive/YouTube_Analytics"):
        target_dirs.append("/content/drive/MyDrive/YouTube_Analytics")
    
    for d in set(target_dirs):
        try:
            os.makedirs(d, exist_ok=True)
            path = os.path.join(d, CACHE_FILE_NAME)
            with open(path, 'w', encoding='utf-8') as f:
                json.dump(cache_data, f, indent=2, ensure_ascii=False)
        except Exception as e:
            print(f"⚠️ Could not save LLM cache to {d}: {e}")

def call_gemini_with_retry(prompt: str, cache_key: str = None, json_mode: bool = True) -> str:
    cache = load_llm_cache() if CONFIG["USE_CACHE"] else {}
    if cache_key and cache_key in cache:
        print(f"⚡ [CACHE HIT] Restoring response for key: {cache_key[:30]}...")
        return cache[cache_key]
    
    if not GOOGLE_API_KEY:
        raise ValueError("CRITICAL: GOOGLE_API_KEY environment variable or Colab secret missing. Cannot proceed with LLM query.")
    
    try:
        from google import genai
        from google.genai import types
        client = genai.Client(api_key=GOOGLE_API_KEY)
    except Exception as e:
        raise RuntimeError(f"Failed to initialize google-genai client: {e}")
    
    max_retries = CONFIG["MAX_RETRIES"]
    backoff = CONFIG["RETRY_BACKOFF_FACTOR"]
    
    for attempt in range(1, max_retries + 1):
        try:
            gen_config = {}
            if json_mode:
                gen_config["response_mime_type"] = "application/json"
            
            response = client.models.generate_content(
                model=CONFIG["GEMINI_MODEL"],
                contents=prompt,
                config=types.GenerateContentConfig(**gen_config) if gen_config else None
            )
            
            res_text = response.text.strip()
            if not res_text:
                raise ValueError("Received empty response string from Gemini API.")
            
            if cache_key and CONFIG["USE_CACHE"]:
                cache[cache_key] = res_text
                save_llm_cache(cache)
            
            return res_text
        except Exception as err:
            print(f"⚠️ LLM API Attempt {attempt}/{max_retries} failed: {err}")
            if attempt < max_retries:
                sleep_time = backoff * attempt
                print(f"   Retrying in {sleep_time} seconds...")
                time.sleep(sleep_time)
            else:
                raise RuntimeError(f"CRITICAL: Exhausted all {max_retries} retries for Gemini API query. Error details: {err}")

print("✅ LLM client engine and persistent caching functions initialized successfully.")


## 📂 4. Dataset Ingestion & Top 10 Video Isolation per Subscribed Channel

Loads the subscribed channels recent videos dataset exported by `4.Youtube_channel_subscription_clusters.ipynb` (`{Channel_Name}_subscribed_channels_recent_videos.json` or `.csv`). For each unique channel, filters and ranks its videos to extract the **10 best performing videos** (ranked by `view_count` or engagement), or 10 recent videos if metrics are unavailable.


In [ ]:
safe_channel_id = sanitize_filename(CONFIG["CHANNEL_IDENTIFIER"])
search_dirs = [ACTIVE_OUTPUT_DIR, "/content/drive/MyDrive/YouTube_Analytics", "data", "analysis", "."]

subscribed_videos_file = None
subscribed_vids_data = []

for d in search_dirs:
    if not os.path.exists(d):
        continue
    candidate_json = os.path.join(d, f"{safe_channel_id}_subscribed_channels_recent_videos.json")
    candidate_csv = os.path.join(d, f"{safe_channel_id}_subscribed_channels_recent_videos.csv")
    
    if os.path.exists(candidate_json):
        subscribed_videos_file = candidate_json
        with open(candidate_json, 'r', encoding='utf-8') as f:
            subscribed_vids_data = json.load(f)
        print(f"✅ Loaded subscribed channels videos from JSON: {candidate_json}")
        break
    elif os.path.exists(candidate_csv):
        subscribed_videos_file = candidate_csv
        df_csv = pd.read_csv(candidate_csv)
        subscribed_vids_data = df_csv.to_dict(orient="records")
        print(f"✅ Loaded subscribed channels videos from CSV: {candidate_csv}")
        break

if not subscribed_vids_data:
    print(f"⚠️ No dataset found matching pattern '{safe_channel_id}_subscribed_channels_recent_videos.json/csv'. Creating illustrative mock container for schema validation.")
    subscribed_vids_data = []

# Group videos by Channel ID / Channel Title
channels_map = {}
for vid in subscribed_vids_data:
    ch_id = vid.get("channel_id") or vid.get("channel_title") or "Unknown_Channel"
    ch_title = vid.get("channel_title") or ch_id
    if ch_id not in channels_map:
        channels_map[ch_id] = {
            "channel_id": ch_id,
            "channel_title": ch_title,
            "videos": []
        }
    channels_map[ch_id]["videos"].append(vid)

print(f"📊 Total unique channels discovered: {len(channels_map)}")

# Isolate Top 10 best performing (or recent) videos per channel
channel_top_videos = {}
for ch_id, ch_data in channels_map.items():
    vids = ch_data["videos"]
    # Sort by view_count if available, otherwise preserve order
    has_views = any("view_count" in v and v["view_count"] is not None for v in vids)
    if has_views:
        sorted_vids = sorted(vids, key=lambda x: int(x.get("view_count") or 0), reverse=True)
    else:
        sorted_vids = vids
    
    top_10 = sorted_vids[:CONFIG["TOP_VIDEOS_PER_CHANNEL"]]
    channel_top_videos[ch_id] = {
        "channel_id": ch_id,
        "channel_title": ch_data["channel_title"],
        "top_videos": top_10,
        "video_titles": [v.get("video_title") or v.get("title") or "" for v in top_10 if (v.get("video_title") or v.get("title"))]
    }

print(f"✅ Processed top {CONFIG['TOP_VIDEOS_PER_CHANNEL']} videos for {len(channel_top_videos)} channels.")


## 🎯 5. Ingest Channel Description & Iterative Batch Filtering Engine

Ingests the target channel contextual description from `context/channel_description.txt`. Implements the iterative filtering pipeline:
1. **Partition**: Split candidate channel pool into batches of 10 channels.
2. **LLM Evaluation**: Call Gemini 3.1 Flash-Lite to pick the 5 most relevant channels per batch according to the target channel context.
3. **Shuffle & Repeat**: Randomly shuffle surviving channels and repeat batching until total remaining channels $\le 25$.
4. **Final Selection**: Execute final Gemini query on the $\le 25$ candidate pool to extract and rank the Top 10 channels with qualitative reasoning.


In [ ]:
# Load target channel context description
CHANNEL_DESC_PATH = "context/channel_description.txt"
channel_description_text = ""

if os.path.exists(CHANNEL_DESC_PATH):
    with open(CHANNEL_DESC_PATH, "r", encoding="utf-8") as f:
        channel_description_text = f.read().strip()
    print(f"✅ Target channel description loaded successfully ({len(channel_description_text)} characters).")
else:
    print(f"⚠️ Warning: {CHANNEL_DESC_PATH} not found. Utilizing default fallback context.")
    channel_description_text = "Casa Siete explores love, relationships, personal growth, and astrology to foster deeper, conscious human connections."

def filter_batch_with_llm(batch_channels: list, target_desc: str, round_num: int, batch_num: int) -> list:
    if len(batch_channels) <= CONFIG["RETENTION_PER_BATCH"]:
        return [c["channel_id"] for c in batch_channels]
    
    # Construct batch summary payload for LLM prompt
    channels_payload = []
    for idx, c in enumerate(batch_channels, 1):
        titles_str = "
".join([f"    - {t}" for t in c["video_titles"][:10]])
        channels_payload.append(f"Channel [{idx}] ID: {c['channel_id']} | Title: {c['channel_title']}
  Top Video Titles:
{titles_str}")
    
    batch_str = "

".join(channels_payload)
    
    prompt = f"""You are an expert YouTube Media Strategist and Audience Alignment Analyst.

TARGET CHANNEL CONTEXT & THEMATIC MISSION:
{target_desc}

TASK:
Below is a list of {len(batch_channels)} YouTube channels subscribed to by audience members. For each channel, its top performing video titles are provided.
Analyze the thematic relevance, audience overlap, and topical synergy of each candidate channel compared to the target channel context.

Select EXACTLY the top {CONFIG['RETENTION_PER_BATCH']} most relevant channels from this batch that best match or complement the target channel's themes (astrology, love, relationships, emotional wellness, personal transformation, human connection).

Return your response strictly as a JSON object with this structure:
{{
  "selected_channel_ids": ["ch_id_1", "ch_id_2", "ch_id_3", "ch_id_4", "ch_id_5"],
  "brief_reasoning": "Explanation of why these channels were selected."
}}

CANDIDATE CHANNELS BATCH:
{batch_str}
"""
    
    cache_key = f"filter_round_{round_num}_batch_{batch_num}_" + "_".join(sorted([c['channel_id'] for c in batch_channels]))
    res_raw = call_gemini_with_retry(prompt, cache_key=cache_key, json_mode=True)
    
    try:
        data = json.loads(res_raw)
        selected_ids = data.get("selected_channel_ids", [])
        # Validate selected IDs belong to current batch
        valid_batch_ids = {c["channel_id"] for c in batch_channels}
        filtered_ids = [cid for cid in selected_ids if cid in valid_batch_ids]
        
        # If LLM returned fewer than retention count or invalid IDs, fill from remaining batch
        if len(filtered_ids) < CONFIG["RETENTION_PER_BATCH"]:
            for c in batch_channels:
                if c["channel_id"] not in filtered_ids:
                    filtered_ids.append(c["channel_id"])
                if len(filtered_ids) == CONFIG["RETENTION_PER_BATCH"]:
                    break
        return filtered_ids[:CONFIG["RETENTION_PER_BATCH"]]
    except Exception as parse_err:
        print(f"⚠️ Error parsing LLM response in Round {round_num} Batch {batch_num}: {parse_err}. Retaining first {CONFIG['RETENTION_PER_BATCH']} channels.")
        return [c["channel_id"] for c in batch_channels[:CONFIG["RETENTION_PER_BATCH"]]]

# Execution of Iterative Filtering Pipeline
current_pool = list(channel_top_videos.values())
round_history = []
round_counter = 1

print(f"🚀 Starting Iterative Filtering Pipeline with initial pool of {len(current_pool)} channels...")

while len(current_pool) > CONFIG["FINAL_STAGE_THRESHOLD"]:
    print(f"
🔄 --- Round {round_counter}: Pool Size = {len(current_pool)} Channels ---")
    round_history.append({"round": round_counter, "pool_size": len(current_pool)})
    
    # Batch into chunks of 10
    batches = [current_pool[i:i + CONFIG["BATCH_SIZE"]] for i in range(0, len(current_pool), CONFIG["BATCH_SIZE"])]
    surviving_ids = set()
    
    for b_idx, batch in enumerate(batches, 1):
        retained_ids = filter_batch_with_llm(batch, channel_description_text, round_counter, b_idx)
        surviving_ids.update(retained_ids)
        print(f"  - Batch {b_idx}/{len(batches)} ({len(batch)} channels) -> Retained {len(retained_ids)} channels.")
    
    # Filter current pool to survivors
    current_pool = [c for c in current_pool if c["channel_id"] in surviving_ids]
    
    # Shuffle for next iteration to prevent order bias across rounds
    random.shuffle(current_pool)
    round_counter += 1

round_history.append({"round": round_counter, "pool_size": len(current_pool)})
print(f"
✅ Iterative reduction completed! Reached final stage pool of {len(current_pool)} channels (<= {CONFIG['FINAL_STAGE_THRESHOLD']}).")


## 🏆 6. Final Stage Top-10 Relevance Selection & Synthesis

Executes the final LLM selection over the remaining $\le 25$ candidate channel pool. Gemini 3.1 Flash-Lite evaluates all remaining channels holistically against `context/channel_description.txt`, outputting the **Top 10 channels** ranked from 1 to 10 with detailed relevance justifications, thematic score tags, and content synergy analysis.


In [ ]:
def select_final_top_10(final_pool: list, target_desc: str) -> list:
    channels_payload = []
    for idx, c in enumerate(final_pool, 1):
        titles_str = "
".join([f"    - {t}" for t in c["video_titles"][:10]])
        channels_payload.append(f"Channel [{idx}] ID: {c['channel_id']} | Title: {c['channel_title']}
  Top Video Titles:
{titles_str}")
    
    pool_str = "

".join(channels_payload)
    
    prompt = f"""You are a Senior Audience Intelligence & YouTube Media Director.

TARGET CHANNEL CONTEXT & THEMATIC MISSION:
{target_desc}

TASK:
Below is the refined pool of {len(final_pool)} YouTube channels that have survived prior filtering rounds.
Perform a thorough, holistic relevance evaluation comparing each channel's top video titles against the target channel's mission (astrology, love, relationships, emotional growth, conscious connections, self-improvement).

Select and rank EXACTLY the TOP 10 most relevant channels in order from Rank 1 (Most Relevant) to Rank 10.

Return your response strictly as a JSON object formatted as follows:
{{
  "top_channels": [
    {{
      "rank": 1,
      "channel_id": "ch_id_here",
      "channel_title": "Channel Title Here",
      "relevance_score": 98,
      "thematic_alignment": "Core topic match (e.g., Astrology & Relationship Insights)",
      "relevance_justification": "Detailed 2-3 sentence explanation of why this channel is highly relevant to Casa Siete's audience."
    }}
  ],
  "holistic_summary": "Summary of the cross-audience thematic patterns discovered across these top 10 channels."
}}

FINAL CANDIDATE POOL:
{pool_str}
"""
    
    cache_key = "final_top_10_selection_" + "_".join(sorted([c['channel_id'] for c in final_pool]))
    res_raw = call_gemini_with_retry(prompt, cache_key=cache_key, json_mode=True)
    
    try:
        data = json.loads(res_raw)
        top_list = data.get("top_channels", [])
        holistic_summary = data.get("holistic_summary", "")
        return top_list, holistic_summary
    except Exception as parse_err:
        print(f"⚠️ Error parsing final top 10 LLM response: {parse_err}. Extracting top {CONFIG['FINAL_TOP_K']} from pool.")
        fallback_top = []
        for idx, c in enumerate(final_pool[:CONFIG["FINAL_TOP_K"]], 1):
            fallback_top.append({
                "rank": idx,
                "channel_id": c["channel_id"],
                "channel_title": c["channel_title"],
                "relevance_score": max(50, 100 - (idx * 4)),
                "thematic_alignment": "Audience Synergy",
                "relevance_justification": f"Selected during final reduction stage for thematic relevance."
            })
        return fallback_top, "Fallback holistic summary due to parse exception."

print("🎯 Running final stage Top-10 channel selection query...")
final_top_10_results, final_holistic_summary = select_final_top_10(current_pool, channel_description_text)

print(f"✅ Final Top 10 Channels Selected Successfully!")
for ch in final_top_10_results:
    print(f"  #{ch.get('rank')} | Score: {ch.get('relevance_score')} | Title: {ch.get('channel_title')} ({ch.get('channel_id')})")


## 📄 7. Dual-Payload Data Export & Markdown Report Synthesis

Formats the final 10 channels and round reduction metrics into structured DataFrames and exports dual datasets (`.csv`, `.json`) and a comprehensive Markdown executive report (`.md`) to both local disk and Google Drive (`/content/drive/MyDrive/YouTube_Analytics/`).


In [ ]:
safe_channel_id = sanitize_filename(CONFIG["CHANNEL_IDENTIFIER"])

# Prepare export records
export_data = []
for item in final_top_10_results:
    ch_id = item.get("channel_id")
    top_vids = channel_top_videos.get(ch_id, {}).get("top_videos", [])
    export_data.append({
        "rank": item.get("rank"),
        "channel_id": ch_id,
        "channel_title": item.get("channel_title"),
        "relevance_score": item.get("relevance_score"),
        "thematic_alignment": item.get("thematic_alignment"),
        "relevance_justification": item.get("relevance_justification"),
        "top_video_sample_titles": [v.get("video_title") or v.get("title") or "" for v in top_vids[:5]]
    })

df_export = pd.DataFrame(export_data)

# File Paths
csv_filename = f"{safe_channel_id}_top_10_relevant_channels.csv"
json_filename = f"{safe_channel_id}_top_10_relevant_channels.json"
md_filename = f"{safe_channel_id}_top_10_relevant_channels.md"

target_dirs = [ACTIVE_OUTPUT_DIR]
if os.path.exists("/content/drive/MyDrive/YouTube_Analytics"):
    target_dirs.append("/content/drive/MyDrive/YouTube_Analytics")

for d in set(target_dirs):
    os.makedirs(d, exist_ok=True)
    
    # 1. Export CSV
    csv_path = os.path.join(d, csv_filename)
    df_export.to_csv(csv_path, index=False, encoding='utf-8')
    
    # 2. Export JSON
    json_path = os.path.join(d, json_filename)
    payload = {
        "channel_identifier": CONFIG["CHANNEL_IDENTIFIER"],
        "generated_at": datetime.datetime.now(datetime.timezone.utc).isoformat(),
        "total_initial_channels": len(channel_top_videos),
        "final_top_k": len(export_data),
        "round_reduction_history": round_history,
        "holistic_summary": final_holistic_summary,
        "top_10_relevant_channels": export_data
    }
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)
        
    # 3. Export Markdown Report
    md_path = os.path.join(d, md_filename)
    md_lines = [
        f"# 🎯 Top 10 Most Relevant Subscribed Channels Report",
        f"**Target Channel**: `{CONFIG['CHANNEL_IDENTIFIER']}`  ",
        f"**Generated**: {datetime.datetime.now(datetime.timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}  ",
        f"**Initial Pool Size**: {len(channel_top_videos)} channels | **Final Top Selected**: 10 channels
",
        f"---",
        f"## 📌 Holistic Audience & Content Alignment Summary
",
        f"{final_holistic_summary}
",
        f"---",
        f"## 🏆 Top 10 Selected Channels
"
    ]
    
    for ch in export_data:
        md_lines.append(f"### #{ch['rank']} - {ch['channel_title']} (Score: {ch['relevance_score']}/100)")
        md_lines.append(f"- **Channel ID**: `{ch['channel_id']}`")
        md_lines.append(f"- **Thematic Alignment**: {ch['thematic_alignment']}")
        md_lines.append(f"- **Relevance Justification**: {ch['relevance_justification']}")
        md_lines.append(f"- **Sample Top Videos**:")
        for t in ch['top_video_sample_titles']:
            md_lines.append(f"  - {t}")
        md_lines.append("
" + "-"*40 + "
")
        
    with open(md_path, 'w', encoding='utf-8') as f:
        f.write("
".join(md_lines))

print(f"✅ Successfully exported dual dataset payloads and Markdown summary report across target directories:")
print(f"  - CSV: {csv_filename}")
print(f"  - JSON: {json_filename}")
print(f"  - Markdown: {md_filename}")


## 📈 8. Visual Analytics & Filtering Trajectory Projections

Generates high-resolution inline analytics charts using Seaborn and Matplotlib:
1. **Iterative Channel Survival Trajectory**: Line and bar chart tracking channel pool size reduction across iterative batch filtering rounds ($N_0 	o N_1 \dots 	o 25 	o 10$).
2. **Top 10 Relevance Score Distribution**: Horizontal bar chart visualizing the assigned qualitative relevance scores across the top 10 selected channels.


In [ ]:
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({'font.size': 10, 'figure.titlesize': 14})

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Chart 1: Iterative Channel Survival Trajectory
if round_history:
    df_rounds = pd.DataFrame(round_history)
    sns.lineplot(data=df_rounds, x="round", y="pool_size", ax=axes[0], marker="o", color="#2b5c8f", linewidth=2.5, markersize=8)
    axes[0].set_title("Iterative Batch Filtering Survival Trajectory", fontsize=12, fontweight="bold")
    axes[0].set_xlabel("Filtering Round", fontsize=11)
    axes[0].set_ylabel("Remaining Channel Pool Size", fontsize=11)
    for idx, row in df_rounds.iterrows():
        axes[0].annotate(f"{int(row['pool_size'])}", (row['round'], row['pool_size']),
                         textcoords="offset points", xytext=(0,10), ha='center', fontweight="bold")

# Chart 2: Top 10 Selected Channels Relevance Scores
if export_data:
    df_top = pd.DataFrame(export_data).sort_values(by="rank", ascending=False)
    sns.barplot(data=df_top, x="relevance_score", y="channel_title", ax=axes[1], palette="crest")
    axes[1].set_title("Top 10 Channels Qualitative Relevance Score", fontsize=12, fontweight="bold")
    axes[1].set_xlabel("Relevance Score (0 - 100)", fontsize=11)
    axes[1].set_ylabel("Channel Title", fontsize=11)
    for p in axes[1].patches:
        width = p.get_width()
        axes[1].annotate(f"{width:.0f}", (width - 5, p.get_y() + p.get_height() / 2.),
                         ha='center', va='center', color='white', fontweight='bold', fontsize=10)

plt.tight_layout()
plot_path = os.path.join(ACTIVE_OUTPUT_DIR, f"{safe_channel_id}_relevance_filtering_analytics.png")
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
plt.show()

print(f"📊 Visual analytics charts rendered inline and saved to {plot_path}")
